<a href="https://colab.research.google.com/github/Rakshit-Gupta-77/Machine-Learning-Journey/blob/main/web_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import libraries


In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np

## 2. Understand `requests.get()`

`requests.get(url)` downloads a webpage. `.text` gives us the HTML returned by the server.

In [2]:
url = 'https://books.toscrape.com/'

response = requests.get(url)

print('Status code:', response.status_code)
print(response.text[:1000])

Status code: 200
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="stat

## 3. User-Agent headers

A User-Agent identifies the client making the request. Some websites use it as part of their access controls.

In [3]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)
webpage = response.text

print('Status code:', response.status_code)

Status code: 200


## 4. Parse HTML with BeautifulSoup


In [4]:
soup = BeautifulSoup(webpage, 'html.parser')

print(soup.title.text)


    All products | Books to Scrape - Sandbox



## 5. Find all books


In [5]:
books = soup.find_all('article', class_='product_pod')

print('Number of books:', len(books))

Number of books: 20


## 6. Extract book information


In [6]:
name = []
price = []
rating = []

for book in books:
    name.append(book.h3.a['title'])
    price.append(book.find('p', class_='price_color').text.strip())
    rating.append(book.find('p', class_='star-rating')['class'][1])

print(name[:5])
print(price[:5])
print(rating[:5])

['A Light in the Attic', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History of Humankind']
['Â£51.77', 'Â£53.74', 'Â£50.10', 'Â£47.82', 'Â£54.23']
['Three', 'One', 'One', 'Four', 'Five']


## 7. Create a DataFrame


In [7]:
df = pd.DataFrame({
    'name': name,
    'price': price,
    'rating': rating
})

df.head()

,name,price,rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five


## 8. Scrape multiple pages

Instead of the removed `DataFrame.append()`, modern Pandas uses `pd.concat()`.

In [8]:
final = pd.DataFrame()

for page in range(1, 4):
    url = f'https://books.toscrape.com/catalogue/page-{page}.html'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    books = soup.find_all('article', class_='product_pod')

    name = []
    price = []
    rating = []

    for book in books:
        name.append(book.h3.a['title'])
        price.append(book.find('p', class_='price_color').text.strip())
        rating.append(book.find('p', class_='star-rating')['class'][1])

    page_df = pd.DataFrame({
        'name': name,
        'price': price,
        'rating': rating
    })

    # Modern replacement for: final.append(page_df, ignore_index=True)
    final = pd.concat([final, page_df], ignore_index=True)

print(final.shape)
final.head()

(60, 3)


,name,price,rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five


## 9. Save the scraped data as CSV


In [9]:
final.to_csv('books_scraped.csv', index=False)
print('CSV saved successfully!')

CSV saved successfully!
